# 🌐 Aula 06 — Web Scraping

## 🕸️ Extração de dados da Web para Mineração de Dados

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Requests, BeautifulSoup e Pandas

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de Web Scraping;
- Identificar quando a raspagem de dados pode ser utilizada;
- Fazer requisições HTTP com Python;
- Interpretar uma página HTML;
- Localizar informações utilizando BeautifulSoup;
- Extrair textos e tabelas;
- Transformar dados extraídos em DataFrames;
- Realizar uma etapa simples de limpeza após a extração;
- Compreender cuidados éticos e técnicos ao coletar dados da Web;
- Relacionar Web Scraping ao processo ETL;
- Planejar uma possível fonte Web para o próprio projeto.

> **Projeto didático:** nesta aula vamos utilizar dados públicos de produtos de uma loja fictícia criada dentro do próprio notebook. O projeto do motor elétrico continua sendo nosso exemplo principal de Mineração de Dados, mas a técnica será demonstrada em um cenário de coleta Web.


# 🌐 1. De onde vêm os dados?

Na Aula 5 aprendemos que uma etapa do ETL é o **Extract**.

Os dados podem vir de:

- arquivos CSV;
- Excel;
- bancos de dados;
- APIs;
- sensores;
- sistemas corporativos;
- páginas da Web.

Hoje vamos estudar uma técnica utilizada quando precisamos extrair informações apresentadas em páginas HTML:

> **Web Scraping**


# 🕷️ 2. O que é Web Scraping?

Web Scraping é o processo automatizado de coleta de informações disponíveis em páginas da Web.

Imagine uma página contendo:

```text
Produto       Preço       Avaliação
Notebook      3500        4.5
Monitor       1200        4.2
Teclado       300         4.7
```

Em vez de copiar manualmente os dados, podemos utilizar Python para:

```text
Página Web
    ↓
HTML
    ↓
Python
    ↓
Extração
    ↓
DataFrame
    ↓
Análise
```

Isso transforma uma página Web em uma possível fonte de dados para um projeto de Mineração de Dados.


# ⚠️ 3. Web Scraping não é simplesmente "copiar um site"

Antes de coletar dados, precisamos observar:

- se a informação é pública;
- se o site permite a coleta;
- termos de uso;
- robots.txt;
- frequência das requisições;
- volume de dados;
- proteção de dados pessoais;
- direitos autorais;
- existência de uma API oficial.

> **Boa prática:** quando uma API oficial fornece os dados necessários, normalmente ela deve ser preferida ao scraping.

Nesta aula utilizaremos uma página **fictícia criada no notebook**, para que possamos praticar a técnica sem depender de um site externo.


# 💻 4. Preparando o ambiente

Vamos importar as bibliotecas.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 🧪 5. Criando uma página HTML para o laboratório

Vamos criar uma pequena página HTML representando produtos de uma loja.

A ideia é simular o conteúdo que encontraríamos em uma página real.


In [ ]:
html = '<!DOCTYPE html>\n<html>\n<head>\n    <title>Catálogo de Produtos</title>\n</head>\n<body>\n    <h1>Catálogo</h1>\n\n    <div class="produto">\n        <h2>Notebook Pro</h2>\n        <p class="categoria">Computadores</p>\n        <p class="preco">R$ 3.500,00</p>\n        <p class="avaliacao">4.5</p>\n    </div>\n\n    <div class="produto">\n        <h2>Monitor 27</h2>\n        <p class="categoria">Monitores</p>\n        <p class="preco">R$ 1.200,00</p>\n        <p class="avaliacao">4.2</p>\n    </div>\n\n    <div class="produto">\n        <h2>Teclado Mecânico</h2>\n        <p class="categoria">Periféricos</p>\n        <p class="preco">R$ 300,00</p>\n        <p class="avaliacao">4.7</p>\n    </div>\n\n    <div class="produto">\n        <h2>Mouse Gamer</h2>\n        <p class="categoria">Periféricos</p>\n        <p class="preco">R$ 180,00</p>\n        <p class="avaliacao">4.6</p>\n    </div>\n</body>\n</html>'

print(html[:500])

# 🔎 6. Entendendo o HTML

HTML possui uma estrutura baseada em **tags**.

Exemplo:

```html
<h2>Notebook Pro</h2>
<p class="preco">R$ 3.500,00</p>
```

Podemos interpretar:

- `h2` → título;
- `p` → parágrafo;
- `class="preco"` → classe utilizada para identificar o conteúdo.

O BeautifulSoup permite navegar nessa estrutura.


In [ ]:
soup = BeautifulSoup(html, "html.parser")

print(soup.title.text)

# 🧭 7. Localizando elementos

Vamos encontrar todos os elementos que representam produtos.


In [ ]:
produtos = soup.find_all("div", class_="produto")

print("Quantidade de produtos:", len(produtos))

Agora vamos analisar apenas o primeiro produto.


In [ ]:
produto = produtos[0]

print(produto.get_text(" ", strip=True))

Podemos localizar cada informação separadamente.


In [ ]:
nome = produto.find("h2").text
categoria = produto.find("p", class_="categoria").text
preco = produto.find("p", class_="preco").text
avaliacao = produto.find("p", class_="avaliacao").text

print(nome)
print(categoria)
print(preco)
print(avaliacao)

# 🔁 8. Extraindo todos os produtos

Agora vamos transformar a extração em um processo automatizado.



In [ ]:
registros = []

for produto in produtos:
    registros.append({
        "produto": produto.find("h2").get_text(strip=True),
        "categoria": produto.find("p", class_="categoria").get_text(strip=True),
        "preco": produto.find("p", class_="preco").get_text(strip=True),
        "avaliacao": produto.find("p", class_="avaliacao").get_text(strip=True)
    })

registros

Transformando os registros em DataFrame:


In [ ]:
df_produtos = pd.DataFrame(registros)

df_produtos

# 🧹 9. Limpando os dados extraídos

Os dados vieram como texto.

Por exemplo:

```text
R$ 3.500,00
```

Para fazer cálculos, precisamos transformar o preço em número.



In [ ]:
df_produtos["preco"] = (
    df_produtos["preco"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_produtos

Agora podemos calcular estatísticas.


In [ ]:
print("Preço médio:", df_produtos["preco"].mean())
print("Preço máximo:", df_produtos["preco"].max())
print("Preço mínimo:", df_produtos["preco"].min())

A avaliação também pode ser convertida para número.


In [ ]:
df_produtos["avaliacao"] = df_produtos["avaliacao"].astype(float)

df_produtos.dtypes

# 🔗 10. Web Scraping dentro do ETL

Observe como a aula anterior se conecta com a atual:

```text
EXTRACT
   ↓
Web Scraping
   ↓
HTML
   ↓
BeautifulSoup
   ↓
DataFrame
   ↓
TRANSFORM
   ↓
limpeza
padronização
conversão
   ↓
LOAD
   ↓
CSV / Banco de Dados
```

Web Scraping é, portanto, uma possível **estratégia de extração** dentro de um processo ETL.


# 🌍 11. Fazendo uma requisição HTTP

Em uma situação real, uma página pode ser acessada por uma URL.

O `requests` permite fazer uma requisição HTTP.

Exemplo:

```python
resposta = requests.get("https://exemplo.com")
```

Vamos entender primeiro o que recebemos de uma requisição.


In [ ]:
print("Biblioteca requests pronta para requisições HTTP.")

### Exemplo conceitual

```python
url = "https://exemplo.com"

resposta = requests.get(url, timeout=10)

print(resposta.status_code)
```

Alguns códigos comuns:

| Código | Significado |
|---|---|
| 200 | Requisição realizada com sucesso |
| 404 | Página não encontrada |
| 403 | Acesso proibido |
| 500 | Erro no servidor |

Em projetos reais, sempre devemos tratar erros e evitar fazer requisições excessivas.


# 🧠 12. Seletores HTML

Para fazer scraping precisamos saber **onde está a informação**.

Podemos utilizar:

```python
find()
find_all()
select()
select_one()
```

Exemplo:

```python
soup.select(".produto")
```

O ponto `.` indica uma classe CSS.

Podemos selecionar:

```text
.produto
.preco
.categoria
```

Isso permite localizar grupos específicos de elementos.


In [ ]:
produtos_css = soup.select(".produto")

len(produtos_css)

In [ ]:
precos = soup.select(".preco")

[preco.get_text(strip=True) for preco in precos]

# 📝 13. Exercícios

## Exercício 1 — Estrutura HTML

Explique o que representam:

```text
<h2>
<p>
class
<div>
```


In [ ]:
# Sua resposta



## Exercício 2 — BeautifulSoup

Crie um objeto `BeautifulSoup` a partir da variável `html` e mostre o título da página.


In [ ]:
# Sua resposta



## Exercício 3 — Quantidade

Conte quantos produtos existem na página.


In [ ]:
# Sua resposta



## Exercício 4 — Extração

Extraia somente os nomes dos produtos e armazene-os em uma lista.


In [ ]:
# Sua resposta



## Exercício 5 — Categorias

Extraia todas as categorias.

Depois descubra quantos produtos existem em cada categoria.


In [ ]:
# Sua resposta



## Exercício 6 — DataFrame

Crie um DataFrame contendo:

- produto;
- categoria;
- preço;
- avaliação.



In [ ]:
# Sua resposta



## Exercício 7 — Limpeza

Converta:

- preço para `float`;
- avaliação para `float`.

Depois mostre os tipos das colunas.


In [ ]:
# Sua resposta



## Exercício 8 — Análise

Qual é:

- o produto mais caro?
- o produto mais barato?
- a avaliação média?
- a maior avaliação?



In [ ]:
# Sua resposta



## Exercício 9 — Seletores

Utilize `select()` para:

1. encontrar todos os produtos;
2. encontrar todos os preços;
3. encontrar todas as avaliações.



In [ ]:
# Sua resposta



## Exercício 10 — ETL

Represente o processo realizado nesta aula:

```text
?????????
   ↓
?????????
   ↓
?????????
   ↓
DataFrame
   ↓
?????????
```

Identifique as etapas de extração e transformação.


In [ ]:
# Sua resposta



# 🔎 14. Desafio — Criando seu próprio catálogo

Modifique o HTML da aula e acrescente **pelo menos 5 novos produtos**.

Cada produto deverá possuir:

- nome;
- categoria;
- preço;
- avaliação.

Depois:

1. faça o scraping;
2. crie o DataFrame;
3. limpe os preços;
4. converta as avaliações;
5. descubra o produto mais caro;
6. descubra a avaliação média por categoria.



In [ ]:
# Desenvolva seu desafio aqui.



# 🚀 15. Aplicação no seu projeto

Agora pense no projeto que seu grupo está desenvolvendo.

Pergunte:

> **Existe alguma informação pública na Web que poderia ajudar a resolver meu problema?**

Exemplos:

- preços;
- indicadores;
- informações de produtos;
- dados de empresas;
- dados meteorológicos;
- informações públicas;
- avaliações;
- notícias;
- catálogos;
- dados de mercado.

Preencha:

| Item | Resposta |
|---|---|
| Problema do projeto | ... |
| Existe informação na Web? | ... |
| Qual informação? | ... |
| Qual seria a fonte? | ... |
| Seria melhor usar API ou scraping? | ... |
| Que cuidados seriam necessários? | ... |

> **Importante:** você não precisa realizar a coleta do projeto agora. O objetivo é identificar se a Web pode ser uma fonte de dados útil.


In [ ]:
# Planejamento



# ⚠️ 16. Boas práticas

Ao realizar Web Scraping:

- respeite os termos de uso;
- verifique se existe API oficial;
- consulte as regras do site;
- não faça milhares de requisições desnecessárias;
- utilize intervalos quando apropriado;
- não tente contornar mecanismos de proteção;
- evite coletar dados pessoais sem necessidade;
- registre a fonte dos dados;
- registre a data da coleta.

Uma base de dados sem origem documentada perde parte do seu valor científico e analítico.


# 📌 17. Checklist da Aula

- [ ] Sei explicar o que é Web Scraping;
- [ ] Sei diferenciar scraping e API;
- [ ] Entendo a estrutura básica de HTML;
- [ ] Sei utilizar BeautifulSoup;
- [ ] Sei localizar elementos com `find()` e `find_all()`;
- [ ] Sei utilizar seletores CSS com `select()`;
- [ ] Sei transformar dados extraídos em DataFrame;
- [ ] Sei limpar dados extraídos;
- [ ] Entendo como Web Scraping pode fazer parte do ETL;
- [ ] Sei identificar cuidados na coleta de dados da Web.

---

# 🎯 Conclusão

Nas últimas aulas construímos:

```text
Aula 3 → Manipulação
Aula 4 → Limpeza
Aula 5 → ETL
Aula 6 → Web Scraping
```

Agora já sabemos que uma fonte de dados pode estar fora dos nossos arquivos e bancos de dados.

Na próxima aula vamos trabalhar outra fonte fundamental:

> 🗄️ **Banco de Dados, SQL e integração com Pandas.**

Isso permitirá que o aluno trabalhe com bases mais próximas dos ambientes encontrados no mercado.
